In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

In [4]:
# Import CSV and JSON datasets
df_csv = pd.read_csv('Stationary.csv')
df_json = pd.read_json('Stationary.json')

In [5]:
print("CSV Dataset Shape:", df_csv.shape)

CSV Dataset Shape: (1901, 12)


In [6]:
print("JSON Dataset Shape:", df_json.shape)

JSON Dataset Shape: (1901, 12)


In [7]:
print("\nCSV Dataset Preview:")
print(df_csv.head())


CSV Dataset Preview:
   Transaction_ID  Trans Number        Date  Year   Region    Customer_Type  \
0  SO-201901-0001         10001  2019-01-01  2019     West  Online Customer   
1  SO-201901-0002         10002  2019-01-01  2019     West  Online Customer   
2  SO-201901-0003         10003  2019-01-01  2019     East  Retail Customer   
3  SO-201901-0004         10004  2019-01-01  2019    South  Retail Customer   
4  SO-201901-0005         10005  2019-01-02  2019  Central  Retail Customer   

           Category       Product  Units  Unit_Price  Dollars Payment_Method  
0       School Bags      Backpack      6        28.0    168.0       E-Wallet  
1    Paper Products      Notebook      3         3.5     10.5           Cash  
2      Art Supplies   Glue Sticks      5         4.5     22.5           Cash  
3       School Bags      Backpack     15        28.0    420.0           Cash  
4  Writing Supplies  Highlighters      7         6.5     45.5       E-Wallet  


In [8]:
print("\nJSON Dataset Preview:")
print(df_json.head())


JSON Dataset Preview:
   Transaction_ID  Trans Number       Date  Year   Region    Customer_Type  \
0  SO-201901-0001         10001 2019-01-01  2019     West  Online Customer   
1  SO-201901-0002         10002 2019-01-01  2019     West  Online Customer   
2  SO-201901-0003         10003 2019-01-01  2019     East  Retail Customer   
3  SO-201901-0004         10004 2019-01-01  2019    South  Retail Customer   
4  SO-201901-0005         10005 2019-01-02  2019  Central  Retail Customer   

           Category       Product  Units  Unit_Price  Dollars Payment_Method  
0       School Bags      Backpack      6        28.0    168.0       E-Wallet  
1    Paper Products      Notebook      3         3.5     10.5           Cash  
2      Art Supplies   Glue Sticks      5         4.5     22.5           Cash  
3       School Bags      Backpack     15        28.0    420.0           Cash  
4  Writing Supplies  Highlighters      7         6.5     45.5       E-Wallet  


In [9]:
print("\nMissing values in CSV:", df_csv.isnull().sum().sum())


Missing values in CSV: 0


In [10]:
print("Missing values in JSON:", df_json.isnull().sum().sum())

Missing values in JSON: 0


In [11]:
print("\nDuplicate rows in CSV:", df_csv.duplicated().sum())


Duplicate rows in CSV: 0


In [12]:
print("Duplicate rows in JSON:", df_json.duplicated().sum())

Duplicate rows in JSON: 0


In [13]:
# Use CSV as the main dataset
df = df_csv.copy()

In [14]:
# Convert date column to datetime
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

In [15]:
# Handle missing values in important analysis columns
df = df.dropna(subset=['Date', 'Product', 'Region', 'Dollars'])

In [16]:
# Create month and year columns
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year

In [19]:
product_sales = (
    df.groupby('Product', as_index=False)['Dollars']
      .sum()
      .sort_values('Dollars', ascending=False)
)

fig = px.bar(
    product_sales,
    x='Product',
    y='Dollars',
    title='Total Sales by Product',
    labels={'Dollars': 'Total Sales ($)', 'Product': 'Product'},
    text_auto='.2s',
    hover_data={'Dollars': ':,.2f'}
)

fig.update_layout(xaxis_tickangle=-35)
fig.show()

print("Top Selling Product by Revenue:",
      product_sales.iloc[0]['Product'])
print("Highest Product Revenue:",
      product_sales.iloc[0]['Dollars'])


Top Selling Product by Revenue: Backpack
Highest Product Revenue: 105140.0


In [20]:
region_sales = (
    df.groupby('Region', as_index=False)['Dollars']
      .sum()
      .sort_values('Dollars', ascending=False)
)

fig = px.pie(
    region_sales,
    names='Region',
    values='Dollars',
    title='Sales Distribution by Region',
    hole=0.4,
    hover_data={'Dollars': ':,.2f'},
)

fig.update_traces(
    textposition='inside',
    textinfo='label+percent'
)

fig.show()

top_region = region_sales.iloc[0]
print("Highest Revenue Region:", top_region['Region'])
print("Regional Sales:", top_region['Dollars'])


Highest Revenue Region: Central
Regional Sales: 73026.0


In [21]:
# Group sales by year and month
monthly_sales = (
    df.groupby(df['Date'].dt.to_period('M'))['Dollars']
      .sum()
      .reset_index()
)

monthly_sales['Month'] = monthly_sales['Date'].astype(str)

fig = px.line(
    monthly_sales,
    x='Month',
    y='Dollars',
    title='Monthly Sales Trend',
    labels={'Month': 'Year-Month', 'Dollars': 'Total Sales ($)'},
    markers=True,
    hover_data={'Dollars': ':,.2f'}
)

fig.update_layout(
    xaxis_tickangle=-45,
    hovermode='x unified'
)

fig.show()

best_month = monthly_sales.loc[monthly_sales['Dollars'].idxmax()]
print("Highest Sales Month:", best_month['Month'])
print("Highest Monthly Sales:", best_month['Dollars'])


Highest Sales Month: 2019-06
Highest Monthly Sales: 8859.0


In [22]:
print("========== SALES ANALYSIS SUMMARY ==========")

print("Total Revenue: $", round(df['Dollars'].sum(), 2))
print("Total Units Sold:", df['Units'].sum())
print("Number of Transactions:", len(df))

print("\nTop 5 Products by Revenue:")
print(
    df.groupby('Product')['Dollars']
      .sum()
      .sort_values(ascending=False)
      .head(5)
)

print("\nSales by Region:")
print(
    df.groupby('Region')['Dollars']
      .sum()
      .sort_values(ascending=False)
)

print("\nSales by Customer Type:")
print(
    df.groupby('Customer_Type')['Dollars']
      .sum()
      .sort_values(ascending=False)
)


========== SALES ANALYSIS SUMMARY ==========
Total Revenue: $ 325101.0
Total Units Sold: 43017
Number of Transactions: 1901

Top 5 Products by Revenue:
Product
Backpack              105140.0
Stapler                42265.5
Scissors               31122.0
Whiteboard Markers     28896.0
Highlighters           26819.0
Name: Dollars, dtype: float64

Sales by Region:
Region
Central    73026.0
North      71287.0
South      62740.0
West       60836.5
East       57211.5
Name: Dollars, dtype: float64

Sales by Customer Type:
Customer_Type
Wholesale          231651.5
Retail Customer     66024.0
Online Customer     27425.5
Name: Dollars, dtype: float64
